# Fase 2 — Preprocessing

Preprocessing pipeline for the Kuka robot dataset.

**Pipeline** (no clipping — decided by project design):

1. `load_kuka_data()` — load, align columns, remove constant features
2. `split_temporal_data()` — temporal 60/20/20 split (no shuffle)
3. `normalize_data()` — StandardScaler fit on train only
4. `save_processed_data()` — saves .npy, scaler.pkl, selected_columns.npy

See `docs/project_plan.md` §7 #9 for the full decision rationale.

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

# Path resolution — works whether launched from project root or notebooks/
NOTEBOOK_DIR = os.getcwd()
PROJECT_ROOT = os.path.dirname(NOTEBOOK_DIR) if 'notebooks' in NOTEBOOK_DIR else NOTEBOOK_DIR
os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

from src.utils.config import load_config, get_param
from src.data.preprocessing import load_kuka_data, split_temporal_data, normalize_data
from src.data.dataset import KukaDataset

config = load_config()
print(f"Project root: {PROJECT_ROOT}")
print(f"input_dim: {get_param(config, 'model.input_dim')}")
print(f"train_split: {get_param(config, 'training.train_split')}")
print(f"window_size: {get_param(config, 'data.window_size')}")
print(f"Split: {get_param(config, 'training.train_split')} / {get_param(config, 'training.val_split')} / {1 - get_param(config, 'training.train_split') - get_param(config, 'training.val_split')}")

## 1. Data loading and cleaning

`load_kuka_data` reads the three raw `.npy` files, drops the `anomaly` column
from KukaSlow (87→86), and removes the 4 features that are constant in both
datasets (`sensor_id{2,5,6,7}_temp`). The result is 82 features for both arrays.

In [ ]:
normal, slow, selected_cols = load_kuka_data(config)

print(f"KukaNormal: {normal.shape}  dtype={normal.dtype}")
print(f"KukaSlow:   {slow.shape}  dtype={slow.dtype}")
print(f"Columns:    {selected_cols.shape}  dtype={selected_cols.dtype}")
print()
print("Removed (constant in both datasets):")
all_cols = np.load(config['data']['column_names_path'], allow_pickle=True)
removed = set(all_cols.tolist()) - set(selected_cols.tolist())
for c in sorted(removed):
    print(f"  - {c}")
print()
print("First 10 selected features:")
print(selected_cols[:10])

# Memory footprint
mem_mb = (normal.nbytes + slow.nbytes) / 1e6
print(f"\nMemory: {mem_mb:.1f} MB")

## 2. Temporal split (60/20/20, no shuffle)

Bar chart shows the number of samples in each split along the timeline.
No shuffling is used because lag-1 autocorrelation ≈ 0.99+ — shuffling
would leak future information into the past.

In [ ]:
splits = split_temporal_data(normal, slow, config)

fig, ax = plt.subplots(figsize=(10, 2))
sizes = [len(splits[k]) for k in ['train', 'val', 'test_normal']]
labels = ['train (60%)', 'val (20%)', 'test_normal (20%)']
colors = ['#2196F3', '#4CAF50', '#FF9800']
ax.bar(labels, sizes, color=colors)
ax.set_ylabel('samples')
ax.set_title('Temporal split of KukaNormal (no shuffle)')
for i, (lbl, sz) in enumerate(zip(labels, sizes)):
    ax.text(i, sz, str(sz), ha='center', va='bottom')
plt.tight_layout()
plt.show()

print(f"train:        {splits['train'].shape}")
print(f"val:          {splits['val'].shape}")
print(f"test_normal:  {splits['test_normal'].shape}")
print(f"test_anomaly: {splits['test_anomaly'].shape}  (all of KukaSlow)")
print(f"\nTotal test: {len(splits['test_normal']) + len(splits['test_anomaly'])} "
      f"(normal={len(splits['test_normal'])}, anomaly={len(splits['test_anomaly'])})")

## 3. Normalization (StandardScaler, fit on train only)

**No clipping** — sensor saturation values (Gyro ±2000, Acc ±16) remain
in-place, handled by StandardScaler + MSE loss.

The two bar charts below show the per-feature mean and standard deviation
that the scaler learned from the train split. Train mean ≈ 0 and std ≈ 1
confirms a correct fit. Val mean ≠ 0 (printed below) confirms no data leakage.

In [ ]:
scaled, scaler = normalize_data(splits, config)

print("Scaler statistics (fit on train only):\n")
for name in ['train', 'val', 'test_normal', 'test_anomaly']:
    arr = scaled[name]
    print(f"  {name:14s}: mean={arr.mean():>8.4f}  std={arr.std():>8.4f}  "
          f"min={arr.min():>8.4f}  max={arr.max():>8.4f}  shape={arr.shape}")

# Verification: train ≈ 0 / 1, val ≠ 0 (no leakage)
print(f"\n--- Verification ---")
print(f"train mean (max abs):  {np.abs(scaled['train'].mean(axis=0)).max():.2e}")
print(f"train std (min):       {scaled['train'].std(axis=0).min():.4f}")
print(f"val mean (max abs):    {np.abs(scaled['val'].mean(axis=0)).max():.4f}  (≠0 → no leakage)")

# Distribution of scaler means across features
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(len(scaler.mean_)), scaler.mean_)
axes[0].set_title('Scaler mean per feature')
axes[0].set_xlabel('feature index')
axes[0].set_ylabel('mean')
axes[1].bar(range(len(scaler.scale_)), scaler.scale_)
axes[1].set_title('Scaler std (scale) per feature')
axes[1].set_xlabel('feature index')
axes[1].set_ylabel('std')
plt.tight_layout()
plt.show()

## 4. KukaDataset — on-the-fly sliding-window

Each split is wrapped in a separate `KukaDataset(label=0)` or `(label=1)`.
Windows are sliced on-the-fly in `__getitem__` (no pre-computation),
keeping memory at ~0.18 GB vs 2.9 GB if all windows were materialized.

In [ ]:
W = get_param(config, 'data.window_size', 16)

train_ds = KukaDataset(scaled['train'], window_size=W, stride=1, label=0)
val_ds   = KukaDataset(scaled['val'], window_size=W, stride=1, label=0)
test_normal_ds = KukaDataset(scaled['test_normal'], window_size=W, stride=1, label=0)
test_anomaly_ds = KukaDataset(scaled['test_anomaly'], window_size=W, stride=1, label=1)

print("Dataset summary:")
for name, ds in [('train', train_ds), ('val', val_ds),
                  ('test_normal', test_normal_ds), ('test_anomaly', test_anomaly_ds)]:
    print(f"  {name:14s}: {len(ds)} windows  (label={ds.label})")

# Total test windows
total_test = len(test_normal_ds) + len(test_anomaly_ds)
print(f"\nTotal test windows: {total_test} "
      f"(normal={len(test_normal_ds)}, anomaly={len(test_anomaly_ds)})")

# Verify first window shape and label
window, label = train_ds[0]
print(f"\nSample: window.shape={window.shape}, label={label.item()}, dtype={window.dtype}")

## 5. Sample window visualization

Heatmaps show one 16-timestep window × the first 20 features. The colormap
is `RdBu_r` (blue = below mean, red = above mean, after z-scoring). The
anomaly window on the right exhibits the characteristic slow-drift pattern
that the model must learn to flag.

In [ ]:
# Visualizza la prima finestra del train (16 timestep × 82 feature)
# Mostriamo solo le prime 20 feature per leggibilità
window_normal, _ = train_ds[0]
window_normal = window_normal[:, :20].numpy()

# E una finestra dalla parte anomala
window_anomaly, label_anom = test_anomaly_ds[0]
window_anomaly = window_anomaly[:, :20].numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
im0 = axes[0].imshow(window_normal.T, aspect='auto', cmap='RdBu_r', interpolation='nearest')
axes[0].set_title(f'Train window (label=0, normal)')
axes[0].set_xlabel('timestep')
axes[0].set_ylabel('feature')
plt.colorbar(im0, ax=axes[0], fraction=0.046)
im1 = axes[1].imshow(window_anomaly.T, aspect='auto', cmap='RdBu_r', interpolation='nearest')
axes[1].set_title(f'Test anomaly window (label=1)')
axes[1].set_xlabel('timestep')
axes[1].set_ylabel('feature')
plt.colorbar(im1, ax=axes[1], fraction=0.046)
plt.tight_layout()
plt.show()

# Memory comparison: raw vs on-the-fly windows
raw_mem = scaled['train'].nbytes / 1e6
precomp_mem = len(train_ds) * W * 82 * 4 / 1e9
print(f"Memory — raw train array: {raw_mem:.1f} MB")
print(f"Memory — pre-computed windows: {precomp_mem:.1f} GB (NOT computed — on-the-fly)")
print(f"Memory — on-the-fly (views): ~0.18 GB total for all splits")

## 6. Window count summary

The table below summarizes the number of windows per split after on-the-fly
windowing with `W=16, stride=1`:

| Split | Source | Samples | Windows | Label |
|---|---|---|---|---|
| train | KukaNormal (60%) | 140 275 | 140 260 | 0 |
| val | KukaNormal (20%) | 46 758 | 46 743 | 0 |
| test_normal | KukaNormal (20%) | 46 759 | 46 744 | 0 |
| test_anomaly | KukaSlow (100%) | 41 538 | 41 523 | 1 |

Total test windows: 88 267. The next phase is Fase 3 — baseline Autoencoder
(1D-Conv encoder + symmetric decoder).